# BSAN 775 Assignment 1 (Chapters 1 to 6)

**Chapters:** 1 to 6

**Points:** 100

**Due:** Wednesday, October 7, 2026, 11:59 pm Central

Prairie Wholesale's category manager wants to know which product categories
make money, and whether the discounting policy is costing more than it
returns. The orders table records what was sold and at what price. The products
table records what each item cost the company, and neither answers the question
on its own.

You will build the table that does, describe what it contains, test one
relationship inside it, and write the recommendation.

## What to submit

One notebook, downloaded as `.ipynb`, uploaded to the Assignment 1 dropbox in
Blackboard. Before you submit, run **Restart and run all** and confirm the
notebook runs top to bottom with no errors.

## How this is graded

| Problem | Points |
|---|---|
| 1. Build the line-level margin table | 15 |
| 2. Describe the margin rate | 25 |
| 3. Discount against margin | 30 |
| 4. The category table and the recommendation | 30 |

Forty of the hundred points are written interpretation.

A correct number with no interpretation receives part of the marks. The cells
marked **graded** are where the interpretation belongs.

If you used an AI assistant, add a line at the end naming the tool and what it
helped with, per Appendix D.

## Running this notebook

Run the setup cell, then the provided cell, then work down. You enter your work
in the cells marked `# TODO`. Everything else is provided.

**If you are in Google Colab:** this notebook opened read-only from GitHub. Click
**Copy to Drive** in the toolbar (or *File > Save a copy in Drive*) before you
edit anything, then work in that copy. Edits to the read-only original are lost
when the tab closes.

Every step names the function you need. Where a step resembles something you did
in a lab, the lab is named. You are not expected to invent anything.

In [ ]:
# Setup. Run this cell once per session. It installs the packages this
# assignment needs; on Google Colab it also fetches the course data.
%pip install -q pandas plotly
import sys
if "google.colab" in sys.modules:
    !test -d pyba-companion || git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

## Provided: the two tables

`orders` has one row per order **line**. A single order usually has several
lines, one per product. `products` has one row per product, and it carries
`unit_cost`, which is what Prairie Wholesale paid for the item.

In [ ]:
# --- Provided: run this cell second ---
import pandas as pd
import plotly.express as px

from pyba import DATA_DIR

orders   = pd.read_csv(DATA_DIR / "pw_orders.csv", parse_dates=["order_date"])
products = pd.read_csv(DATA_DIR / "pw_products.csv")

print(f"{len(orders):,} order lines, {orders['order_id'].nunique():,} orders")
print(f"{len(products)} products in {products['category'].nunique()} categories")
products.head(3)

---

# Problem 1: build the line-level margin table (15 points)

`line_total` is revenue for that line. It is already in `orders`. The cost is in `products`.

**Most of this table is built in Chapter 3**, in the section "Joining tables",
under the question "which product categories have the best margins?". You are
expected to use it. Points here are low for that reason, and the one column the
chapter does not compute is the basis of the rest of the assignment.

## 1a. Join the cost onto the order lines (4 points)

Merge `products` onto `orders` so that every order line carries its `category`
and its `unit_cost`. Call the result `lines`.

Three details must be right, all of them from Chapter 3:

- Select only the columns you need from `products` before merging:
  `products[["sku", "category", "unit_cost"]]`. Merging the whole table would
  bring a second `unit_price` column across and leave you with `unit_price_x`
  and `unit_price_y`.
- Join on `sku`, with `how="left"` so that no order line is dropped.
- Add `validate="many_to_one"`. Many order lines share one product. If that is
  not true, the merge is silently wrong and this argument raises an error
  instead. Lab 3 used the same argument.


In [ ]:
# TODO (1a): merge the product category and unit cost onto the order lines
lines = None

lines.head(3)

## 1b. Compute the three margin columns (6 points)

Add these to `lines`, in this order:

| Column | Definition |
|---|---|
| `cogs` | `quantity` times `unit_cost`, the cost of goods sold on that line |
| `margin` | `line_total` minus `cogs`, the dollars kept |
| `margin_pct` | `margin` divided by `line_total`, the share of revenue kept |

In Chapter 3 a margin rate is computed for each **category**, from category totals.
`margin_pct` here is different: it is the rate on each individual **line**.
Problems 2 and 3 are about how that line-level rate is distributed and what moves
it, which the chapter does not examine.

Creating a column is `lines["name"] = ...`, as in Chapter 3. These are whole
column operations, so no loop is needed.

**Use `line_total` for revenue.** It is the amount charged, after the discount. Chapter 3 defines `list_value` as `quantity * unit_price`, the line's value at
full price *before* any discount is applied.
If you compute revenue that way here, your margin will ignore every discount in
the data. Nothing will raise an error, and Problems 2, 3 and 4 will all be wrong.

In [ ]:
# TODO (1b): add cogs, margin, and margin_pct
lines["cogs"] = None
lines["margin"] = None
lines["margin_pct"] = None

lines[["line_total", "cogs", "margin", "margin_pct"]].head(3)

## 1c. Check the merge (3 points)

A join can corrupt an analysis without any error being raised, so we check it. Replace each `CHECK_` name below with an
expression that is `True` when the check passes.

- `CHECK_ROW_COUNT`: `lines` has exactly as many rows as `orders`. A left join
  that duplicated rows would fail this.
- `CHECK_NO_MISSING_COST`: no row has a missing `unit_cost`. Use
  `.isna().sum() == 0` on that column. A product in `orders` but absent from
  `products` would fail this.

These names are undefined until you write them, so the cell raises a `NameError`
until you do, which is expected.

In [ ]:
# TODO (1c): replace each name with an expression that is True when the check passes
assert CHECK_ROW_COUNT
assert CHECK_NO_MISSING_COST

print(f"{len(lines):,} lines, margin computed, checks pass")

## 1d. The failure that `validate` prevents (2 points)

In one or two sentences: what would have happened to total revenue if the
products table had contained a duplicate `sku`, and how does `validate` prevent
it?

---

*Replace this line with your answer. This cell is graded.*

---

---

## Provided: a sample for the charts

`lines` has over 140,000 rows. Plotly stores every point it draws inside the
notebook file, so charting all of them would produce a file too large to upload
comfortably. The cell below takes a random 5,000-line sample once, and the charts
in Problems 2 and 3 use it.

`random_state=0` fixes the draw, so your sample is the same every time you run
the notebook and the same as everyone else's. The same call appears in Chapter 5. Summaries and correlations still use the full `lines` table; only the
charts use `chart_sample`.

In [ ]:
# --- Provided: run this after Problem 1 ---
chart_sample = lines.sample(5000, random_state=0)
print(f"{len(chart_sample):,} lines sampled for the charts")

---

# Problem 2: describe the margin rate (25 points)

`margin_pct` is the share of each line's revenue that Prairie Wholesale keeps.
This problem is Chapter 4 applied to a variable you just built.

## 2a. Summarize it (5 points)

One call, `describe()` on the `margin_pct` column, rounded so that the output is
readable. Lab 4 Step 2 asked for the same call.

In [ ]:
# TODO (2a): describe the margin rate

## 2b. Histogram (7 points)

`px.histogram` of `margin_pct` from `chart_sample`, with a bin width you have
chosen.

`px.histogram` takes `nbins`, a **count** of bins, not a width. Decide the width
you want first, then derive the count from it, as you did in Lab 4 Step 3. The
count is the range of the variable divided by the width, rounded to a whole
number, which in code is `nbins = int(round(span / width))`. You compute `span`
from the column's `max()` and `min()`. A width of 0.01, one percentage point, is a
reasonable first try.

Label the axis with `labels={"margin_pct": "Margin rate"}` so that the chart is
readable without explanation. Record the width you chose in a comment.

In [ ]:
# TODO (2b): histogram of margin_pct with a bin width you choose

## 2c. Box plot by category (7 points)

`px.box` on `chart_sample`, with `x="category"` and `y="margin_pct"`.

In Chapter 4 `log_y=True` was used for order totals because those span several
orders of magnitude. A rate between about 0 and 0.5 does not, so leave it off.

In [ ]:
# TODO (2c): box plot of margin rate by category

## 2d. The shape of the distribution (6 points)

Three things, one sentence each:

- The shape of the overall distribution, and which of the mean and the median is
  larger. Either reading of the skew is acceptable if the reasoning is sound.
- Which category has the highest typical margin rate, read from the box plot.
- The mean and the median of `margin_pct` are close together here, unlike the
  order totals in Chapter 4 where they differed by a factor of almost three.
  What does that indicate about this variable?

---

*Replace this line with your answer. This cell is graded.*

---

---

# Problem 3: discount against margin (30 points)

The category manager believes discounting is eroding margin. This problem tests
that claim with the tools of Chapter 5.

## 3a. Scatter plot (6 points)

`px.scatter` with `discount_pct` on the x-axis and `margin_pct` on the y-axis.

Use `chart_sample`, the sample provided above, and set `opacity=0.4` so that
overlapping points stay readable.

In [ ]:
# TODO (3a): scatter of discount_pct against margin_pct, on a sample

## 3b. The pooled correlation (4 points)

One number, the correlation between `discount_pct` and `margin_pct` across all
lines, rounded to three decimals.

The form is `lines["a"].corr(lines["b"]).round(3)`, as in Chapter 5.

In [ ]:
# TODO (3b): the pooled correlation

## 3c. The correlation inside each category (8 points)

A pooled number can differ from the numbers inside the subgroups, so Chapter 5
computes both. Compute the same correlation separately for each `category`.

The chapter's pattern is a loop over the groups:

```
by_cat = {}
for name, g in lines.groupby("category"):
    by_cat[name] = ...
pd.Series(by_cat).round(3)
```

At each pass, `name` is one category and `g` is the sub-table of lines in that
category, so you compute the correlation on `g` as you did on `lines`.

In [ ]:
# TODO (3c): the correlation within each category

## 3d. The claim against the subgroups (12 points)

In three or four sentences:

- State the pooled correlation and its implication for discounting and margin.
- State whether the relationship holds inside every category, and whether the
  subgroup correlations are weaker or stronger than the pooled one.
- In Chapter 5, the pooled correlation between discount and order total **reversed
  sign** inside every channel, and the pooled claim was discarded. Compare what
  you found here to that case. Does the subgroup check overturn your pooled
  result, or support it?
- Say what the category manager should conclude.

---

*Replace this line with your answer. This cell is graded.*

---

---

# Problem 4: the category table and the recommendation (30 points)

## 4a. The category table (5 points)

One table with one row per `category` and three columns:

| Column | Definition |
|---|---|
| `revenue` | total `line_total` |
| `margin` | total `margin` |
| `margin_rate` | total margin divided by total revenue |

Build the first two with a named aggregation, as in Chapter 3 and Lab 3:

```
cat = lines.groupby("category").agg(
    revenue=("line_total", "sum"),
    margin=("margin", "sum"),
)
```

Then add `margin_rate` as a column computed from the two you just made. Note that
this is the rate for the category as a whole, which is not the same as the average
of the line rates you described in Problem 2.

Sort by total margin dollars, descending, so that the table is ordered by each
category's contribution.

This same table is built in Chapter 3, and reproducing it is all that 4a asks.
The work of this problem is in 4b and 4c.

In [ ]:
# TODO (4a): the category table

## 4b. Margin rate by category and channel (5 points for the table, 3 for the reading)

A category's overall rate is a blend across Prairie Wholesale's three channels,
and the category table cannot separate them.

The **pivot table** is introduced in Chapter 5 for this purpose. It is a grid with
one variable's categories down the side and another's across the top, and a
summary statistic in each cell. The chapter's form is

```
lines.pivot_table(values="...", index="...", columns="...", aggfunc="mean")
```

Build one with `margin_pct` as the value, `category` down the side, `channel`
across the top, and the mean as the summary. Round it so that it is readable.

Then answer the question in the graded cell below the table.

In [ ]:
# TODO (4b): pivot of margin rate by category and channel

**4b, the reading (3 points).** Is one channel consistently worse than the
others, and does the pattern hold across every category or only some? One or two
sentences.

---

*Replace this line with your answer. This cell is graded.*

---

## 4c. The category to prioritise (10 points)

The table contains a conflict between two measures. One category earns the
highest margin **rate**, and a different one produces the most margin **dollars**.

In three or four sentences: name both, say which you would put first if the
category manager can only focus on one, and give the reason. There is no single
correct answer, and the marks are for the reasoning.

---

*Replace this line with your answer. This cell is graded.*

---

## 4d. The one-screen dashboard (7 points)

In Chapter 6 a dashboard is built around one question, and the reader should be
able to state its headline within about ten seconds.

The category manager wants one screen. Name the **two** charts you would put on
it, and for each say in one sentence what question it answers. Name the chart type
and what goes on each axis. You do not have to build it.

---

*Replace this line with your answer. This cell is graded.*

---

---

## Before you submit

1. **Restart and run all.** The notebook must run top to bottom with no errors.
2. Check that every cell marked **graded** has your answer in it, not the
   placeholder text.
3. Download as `.ipynb` and upload it to the Assignment 1 dropbox.
4. If you used an AI assistant, add a line below naming the tool and what it
   helped with.

*AI note (if applicable):*